For years, doctors have classified breast cancer into four main types (Basal, HER2-enriched, Luminal A, and Luminal B) using a famous tool called PAM50. PAM50 looks at a specific list of 50 genes to figure out the tumor's type. However, PAM50 was designed for "bulk" sequencing—where you grind up millions of cells together. This new study uses single-cell sequencing, which is notorious for being "sparse" (meaning the data for any one individual cell has a lot of blank spots or missing information). Because of these blank spots, the old PAM50 tool couldn't figure out what type of cancer an individual cell was.

According to the research paper, the researchers created a dataset using the "pseudo-bulk" method. This allowed them to find the special genes for each subtype of breast cancer.

They took cancer cells from the training tumours and did this:

Compared Basal tumours vs others → found 89 genes that are highest in Basal cells → called Basal_SC <br />
Compared HER2E-enriched vs others → found 102 genes → HER2E_SC <br />
Compared Luminal-A vs others → found 46 genes → LumA_SC <br />
Compared Luminal-B vs others → found 65 genes → LumB_SC <br />

The Result. They discovered four brand new lists of genes that act as fingerprints for the four breast cancer types at the single-cell level. They named this new tool SCSubtype.

* First download the official SCSubtype gene list from the link provided in the research paper. This csv file has 4 columns each column representing the list of genes that are important for each type of breast cancer.

In [1]:
import pandas as pd
import numpy as np

# =============================================
# 1. Download the official SCSubtype gene list (from the paper's GitHub)
# =============================================
print("Downloading real SCSubtype signatures...")
url = "https://raw.githubusercontent.com/Swarbricklab-code/BrCa_cell_atlas/main/scSubtype/NatGen_Supplementary_table_S4.csv"
genes_df = pd.read_csv(url)

# Parse the 4 signatures (each column is a list of genes)
signatures = {}
for col in genes_df.columns:
    sig_name = col.replace("Her2E", "HER2E")   # fix spelling
    genes = genes_df[col].dropna().tolist()
    signatures[sig_name] = genes

print("Loaded real signatures:")
for k, v in signatures.items():
    print(f"• {k}: {len(v)} genes (top 5: {v[:5]})")

Loaded real signatures:
• Basal_SC: 89 genes (top 5: ['EMP1', 'TAGLN', 'TTYH1', 'RTN4', 'TK1'])
• HER2E_SC: 102 genes (top 5: ['PSMA2', 'PPP1R1B', 'SYNGR2', 'CNPY2', 'LGALS7B'])
• LumA_SC: 46 genes (top 5: ['SH3BGRL', 'HSPB1', 'PHGR1', 'SOX9', 'CEBPD'])
• LumB_SC: 65 genes (top 5: ['UGCG', 'ARMT1', 'ISOC1', 'GDF15', 'ZFP36'])


We will now apply this new algorithm to determine the type of breast cancer in the sample downloaded from GEO
The Breast cancer data: GSE176078 can be downloaded from GEO[](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE176078)

Here is the description of what was done to obtain these files as per the GEO website.

We performed scRNA-Seq (Chromium, 10X Genomics) on 26 primary tumors from three major clinical subtypes of breast cancer, including 11 ER+, 5 HER2+ and 10 TNBC. We also performed bulk RNA-Seq on a total of 24 of these matched samples.

This is a very big file. Once downloaded, we need to extract the main file into a directory and it will give us multiple tar files one for each sample. There are a total of 26 samples. Each sample will have an ID and will create a new directory when the tar files is unzipped.

Each tar file unzipped directory has 4 files 

count_matrix_barcodes.tsv
count_matrix_genes.tsv
count_matrix_sparse.mtx
metada

We need to load the sparse matrix, convert that into a dense matrix, apply the log1p transformation and pass it to the SCSubtype algorithm.
It will check the genes in each cell type and then vote to find a majority to determine the type of breast cancer.

In [15]:
def run_scsubtype(expr_df, signatures):
        scores = {}
        for st, genes in signatures.items():
            common = [g for g in genes if g in expr_df.index]
            scores[st] = expr_df.loc[common].mean(axis=0) if common else pd.Series(0, index=expr_df.columns)
        score_df = pd.DataFrame(scores)
        return score_df.idxmax(axis=1), score_df.idxmax(axis=1).value_counts().idxmax(), score_df

Once we read the barcodes, genes and the matrix, we can proceed with normalization and running SCSubtype.
We create a pandas DataFrame from the sparse matrix, with genes as rows and barcodes as columns.
Then we apply log normalization and standardization (z-score) to prepare the data for SCSubtype.

In [24]:
import scipy.io as sio
def evaluate_scsubtype_on_real_data(data_dir):
    print(f"Loading real data from {data_dir}...")
    genes = pd.read_csv(data_dir + "/count_matrix_genes.tsv", sep='\t', header=None).iloc[:, 0].values
    print("Read the genes file")

    barcodes = pd.read_csv(data_dir + "/count_matrix_barcodes.tsv", sep='\t', header=None)[0].values
    print("Read the barcodes file")

    matrix = sio.mmread(data_dir + "/count_matrix_sparse.mtx").tocsc()
    print("Read the sparse matrix file")
    
    print(f"✅ Loaded → {len(genes)} genes × {len(barcodes)} cells")

    # Continue with normalization + SCSubtype
    expr = pd.DataFrame.sparse.from_spmatrix(matrix, index=genes, columns=barcodes)
    expr_norm = np.log1p(expr)

    # We have to convert to float64 before standardization to avoid precision issues
    expr_norm = expr_norm.astype(np.float64)
    expr_norm = (expr_norm - expr_norm.mean(axis=0)) / expr_norm.std(axis=0).fillna(0)

    print("Applying SCSubtype on the real data...")

    # Run SCSubtype on the normalized data
    cell_calls, tumour_call, _ = run_scsubtype(expr_norm, signatures)

    print(f"\n🎉 SUCCESS! Whole tumour SCSubtype = **{tumour_call}**")
    print(cell_calls.value_counts())

In [26]:
data_dir_base = "C:/Venky/cancer_gene_expression/breast_cancer_scrnaseq_data/"
samples = ["CID3586", "CID3838", "CID3921", "CID3941", "CID3946"]
for sample in samples:
    print(f"\n=== Evaluating sample {sample} ===")
    data_dir = f"{data_dir_base}/{sample}"
    evaluate_scsubtype_on_real_data(data_dir)


=== Evaluating sample CID3586 ===
Loading real data from C:/Venky/cancer_gene_expression/breast_cancer_scrnaseq_data//CID3586...
Read the genes file
Read the barcodes file
Read the sparse matrix file
✅ Loaded → 29733 genes × 6178 cells
Applying SCSubtype on the real data...

🎉 SUCCESS! Whole tumour SCSubtype = **LumA_SC**
LumA_SC     4049
LumB_SC     2110
HER2E_SC      11
Basal_SC       8
Name: count, dtype: int64

=== Evaluating sample CID3838 ===
Loading real data from C:/Venky/cancer_gene_expression/breast_cancer_scrnaseq_data//CID3838...
Read the genes file
Read the barcodes file
Read the sparse matrix file
✅ Loaded → 29733 genes × 2353 cells
Applying SCSubtype on the real data...

🎉 SUCCESS! Whole tumour SCSubtype = **LumB_SC**
LumB_SC     1369
LumA_SC      975
HER2E_SC       8
Basal_SC       1
Name: count, dtype: int64

=== Evaluating sample CID3921 ===
Loading real data from C:/Venky/cancer_gene_expression/breast_cancer_scrnaseq_data//CID3921...
Read the genes file
Read the bar